# An introduction to training neural networks

<div class="alert alert-block alert-danger">
This lecture gives you ideas of how to apply the gradient descent algorithm to train a simple neural network. This material is not directly examinable but understanding the material may help you with your exam preparations.
</div>

In [ ]:
import time

import numpy as np
from matplotlib import pyplot as plt


plt.style.use("seaborn-v0_8-colorblind")


In [ ]:
# ----------------------------
# Data cache and file names
# ----------------------------
cache_directory = ".mnist_cache"

base_urls = [
    "https://storage.googleapis.com/cvdf-datasets/mnist/",
    "https://ossci-datasets.s3.amazonaws.com/mnist/",
]

training_images_file = "train-images-idx3-ubyte.gz"
training_targets_file = "train-labels-idx1-ubyte.gz"
test_images_file = "t10k-images-idx3-ubyte.gz"
test_targets_file = "t10k-labels-idx1-ubyte.gz"


# ----------------------------
# Download and cache data
# ----------------------------
def data_source():
    return np.lib.npyio.DataSource(cache_directory)


def fetch_bytes(file_name):
    ds = data_source()
    last_error = None

    for base_url in base_urls:
        try:
            with ds.open(base_url + file_name, "rb") as file_handle:
                return file_handle.read()
        except Exception as error:
            last_error = error

    raise RuntimeError(
        "Could not download " + file_name + ". Last error: " + str(last_error)
    )


def parse_idx_images(raw_bytes):
    magic_number = int.from_bytes(raw_bytes[0:4], "big")
    if magic_number != 2051:
        raise ValueError(
            "Invalid image file magic number: " + str(magic_number)
        )

    number_of_images = int.from_bytes(raw_bytes[4:8], "big")
    number_of_rows = int.from_bytes(raw_bytes[8:12], "big")
    number_of_columns = int.from_bytes(raw_bytes[12:16], "big")

    images = np.frombuffer(raw_bytes, dtype=np.uint8, offset=16)
    images = images.reshape(
        number_of_images, number_of_rows * number_of_columns
    )
    images = images.astype(np.float32) / 255.0
    return images


def parse_idx_targets(raw_bytes):
    magic_number = int.from_bytes(raw_bytes[0:4], "big")
    if magic_number != 2049:
        raise ValueError(
            "Invalid target file magic number: " + str(magic_number)
        )

    number_of_targets = int.from_bytes(raw_bytes[4:8], "big")
    targets = np.frombuffer(raw_bytes, dtype=np.uint8, offset=8)

    if targets.shape[0] != number_of_targets:
        raise ValueError("Target count mismatch")

    return targets.astype(np.int64)


def load_mnist():
    training_inputs = parse_idx_images(fetch_bytes(training_images_file))
    training_targets = parse_idx_targets(fetch_bytes(training_targets_file))
    test_inputs = parse_idx_images(fetch_bytes(test_images_file))
    test_targets = parse_idx_targets(fetch_bytes(test_targets_file))

    return training_inputs, training_targets, test_inputs, test_targets


# Load data
training_inputs, training_targets, test_inputs, test_targets = load_mnist()

## Problem

We want to train a neural network to recognise handwritten digits 0-9.

We will solve this using gradient descent to "train" the neural network. We will use the chain rule to find the gradient of a composite function.

We will use a dataset to help us called the MNIST handwritten digit dataset. It contains 70,000 images all labelled with the correct digit.
Each image is a $28 \times 28$ greyscale image, so each image contains 784 pixel values.
The task is to assign the image to one of the ten classes $\{0, 1, 2, 3, 4, 5, 6, 7, 8, 9\}$.

In [ ]:
# Show the first 5 training set images
figure, axes = plt.subplots(1, 5)

for i in range(5):
    image = training_inputs[i].reshape(28, 28)
    axes[i].imshow(image, cmap="gray")
    
    axes[i].set_title(f"Label: {training_targets[i]}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

The input $28 \times 28$ images are "flattened" row-by-row by rearranging the pixel values into a single column vector $\vec{x} \in \mathbb{R}^{784}$.

For output, we want the network to give (approximately) a value 1 in the correct position of a 10 dimensional vector.

We express this mathematically by saying that we want a function:
$$
\vec{\mathcal{F}} \colon \mathbb{R}^{784} \to \mathbb{R}^{10}
$$

The function, we want to optimise should capture the given labels for a subset of all images (called the training set). This will leave further sample labels to test to see how well our method is performing.
The optimisation will be over some parameters (which we define shortly) in the function $\vec{\mathcal{F}}$.
We will call $X$ the set of training images (of size $N$) and suppose we have a labelling function $L \colon X \to \mathbb{R}^{10}$ which takes an image in the training set and gives the correct label.
We will use the notation $L = (L_1, \ldots, L_10)$.

In [ ]:
def indicator_matrix(targets, number_of_classes=10):
    # convert one label per image to a 10 dimensional output for comparison
    matrix = np.zeros((number_of_classes, targets.shape[0]), dtype=np.float32)
    matrix[targets, np.arange(targets.shape[0])] = 1.0
    
    return matrix


In [ ]:
L = indicator_matrix(training_targets)
for i in range(5):
    print(f"image {i} has L = {', '.join(str(L_) for L_ in L[:, i])}")


## The neural network

We will work with what is called a dense neural network with one input layer, one output layer and one hidden layer.
The input layer represents the pixel values in the flattened image (784 values) and the output layer represents the output prediction of the correct label (10 values).
The hidden layer is what brings everything together and we will use $64$ values in this layer.

In [ ]:
def draw_neural_network():
    fig, ax = plt.subplots()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # ------------------------------------------------------------
    # Layer positions
    # ------------------------------------------------------------
    x_input = 0.15
    x_hidden = 0.50
    x_output = 0.85

    # We only draw a few nodes from each layer for clarity
    input_y = np.linspace(0.15, 0.85, 6)
    hidden_y = np.linspace(0.20, 0.80, 5)
    output_y = np.linspace(0.15, 0.85, 10)

    radius = 0.025


    # ------------------------------------------------------------
    # Draw nodes
    # ------------------------------------------------------------
    def draw_nodes(x, ys, colour, edge="black"):
        for y in ys:
            circle = plt.Circle(
                (x, y), radius, facecolor=colour, edgecolor=edge, lw=0.0
            )
            ax.add_patch(circle)

    draw_nodes(x_input, input_y, colour="C0")
    draw_nodes(x_hidden, hidden_y, colour="C1")
    draw_nodes(x_output, output_y, colour="C2")

    # ------------------------------------------------------------
    # Draw connections
    # ------------------------------------------------------------
    for yi in input_y:
        for yh in hidden_y:
            ax.plot(
                [x_input + radius, x_hidden - radius],
                [yi, yh],
                color="gray",
                alpha=0.35,
                lw=0.8,
            )

    for yh in hidden_y:
        for yo in output_y:
            ax.plot(
                [x_hidden + radius, x_output - radius],
                [yh, yo],
                color="gray",
                alpha=0.35,
                lw=0.8,
            )

    # ------------------------------------------------------------
    # Labels for nodes
    # ------------------------------------------------------------
    for i, y in enumerate(input_y):
        if i == 0:
            ax.text(x_input - 0.08, y, r"$x_1$", va="center", fontsize=12)
        elif i == 1:
            ax.text(x_input - 0.08, y, r"$x_2$", va="center", fontsize=12)
        elif i == len(input_y) - 2:
            ax.text(x_input - 0.08, y, r"$x_{783}$", va="center", fontsize=12)
        elif i == len(input_y) - 1:
            ax.text(x_input - 0.08, y, r"$x_{784}$", va="center", fontsize=12)

    # Ellipsis for omitted input nodes
    ax.text(
        x_input - 0.02, 0.50, r"$\vdots$", fontsize=18, ha="center", va="center"
    )

    for i, y in enumerate(hidden_y):
        if i == 0:
            ax.text(
                x_hidden,
                y,
                r"$y_1$",
                ha="center",
                va="center",
                fontsize=10,
                color="white",
            )
        elif i == 1:
            ax.text(
                x_hidden,
                y,
                r"$y_2$",
                ha="center",
                va="center",
                fontsize=10,
                color="white",
            )
        elif i == len(hidden_y) - 2:
            ax.text(
                x_hidden,
                y,
                r"$y_{63}$",
                ha="center",
                va="center",
                fontsize=10,
                color="white",
            )
        elif i == len(hidden_y) - 1:
            ax.text(
                x_hidden,
                y,
                r"$y_{64}$",
                ha="center",
                va="center",
                fontsize=10,
                color="white",
            )

    ax.text(
        x_hidden,
        0.50,
        r"$\vdots$",
        fontsize=18,
        ha="center",
        va="center",
        color="white",
    )

    for i, y in enumerate(output_y):
        ax.text(x_output + 0.05, y, f"$z_{i}$", va="center", fontsize=11)

    # ------------------------------------------------------------
    # Layer titles
    # ------------------------------------------------------------
    ax.text(x_input, 0.93, "Input layer", ha="center", fontsize=12)
    ax.text(x_hidden, 0.93, "Hidden layer", ha="center", fontsize=12)
    ax.text(x_output, 0.93, "Output layer", ha="center", fontsize=12)

    ax.text(x_input, 0.88, "(784 inputs)", ha="center", fontsize=11)
    ax.text(x_hidden, 0.88, "(64 units)", ha="center", fontsize=11)
    ax.text(x_output, 0.88, "(10 outputs)", ha="center", fontsize=11)

    # ------------------------------------------------------------
    # Mathematical labels between layers
    # ------------------------------------------------------------
    ax.text(
        0.32, 0.06, r"$\vec{r} = A \vec{x} + \vec{a}$", ha="center", fontsize=13
    )
    ax.text(
        0.50, 0.02, r"$\vec{y} = \sigma(\vec{r})$", ha="center", fontsize=13
    )

    ax.text(
        0.68, 0.06, r"$\vec{s} = B \vec{y} + \vec{b}$", ha="center", fontsize=13
    )
    ax.text(
        0.85, 0.02, r"$\vec{z} = \sigma(\vec{s})$", ha="center", fontsize=13
    )

    # ------------------------------------------------------------
    # Target / loss annotation
    # ------------------------------------------------------------
    ax.text(
        0.50,
        -0.08,
        # r"Loss for one image: $J = \frac12 \|L - z\|^2$",
        "Loss",
        ha="center",
        fontsize=14,
    )

    plt.tight_layout()
    plt.show()


draw_neural_network()


In [ ]:
def sigma(t):
    # sigmoid activation
    return 1.0 / (1.0 + np.exp(-t))


def forward_map(x, A, a, B, b):
    r = A @ x + a[:, None]
    y = sigma(r)

    s = B @ y + b[:, None]
    
    z = sigma(s)

    return (r, y, s, z)
    

## The objective function

To train the network, we need a quantity to minimise. Since we want the neural network output to match the labels, the simplest choice uses the square of the Euclidean norm. For a single (flattened) image $\vec{x}$ this means
$$
J(\vec{x}) = \frac{1}{2} \| \vec{L}(\vec{x}) - \vec{\mathcal{F}}(\vec{x}) \|^2
$$
For the full training set, we simply average this quantity over all $N$ (flattened) images in $X$.
$$
F = \frac{1}{N} \sum_{\vec{x} \in X} J(\vec{x}) = \frac{1}{2N} \sum_{\vec{x} \in X} \| \vec{L}(\vec{x}) - \vec{\mathcal{F}}(\vec{x}) \|^2.
$$

As we mentioned previously, we want to optimise for parameters in the function $\vec{\mathcal{F}}$. Precisely, these are the values $A$ (size $64 \times 784$), $a$ (size $64$), $B$ (size $10 \times 64$) and $\vec{b}$ (size $10$).
In total this corresponds to $50,890$ parameters so we can say that $F \colon \mathbb{R}^{50\,890} \to \mathbb{R}$.

In [ ]:
def objective(z, targets):
    # objective function F
    L = indicator_matrix(targets, number_of_classes=z.shape[0])
    return 0.5 * np.mean(np.sum((L - z) ** 2, axis=0))


def classification_accuracy(z, targets):
    # measures how many targets we have correct
    predicted_targets = np.argmax(z, axis=0)
    return np.mean(predicted_targets == targets)



def evaluate(x, targets, A, a, B, b
            ):
    _, _, _, z = forward_map(x, A, a, B, b)
    current_objective = objective(z, targets)
    current_accuracy = classification_accuracy(z, targets)
    return current_objective, current_accuracy


## Gradient descent

In [ ]:
# ----------------------------
# Parameter initialisation
# ----------------------------


def initialise_parameters(random_number_generator):
    A = random_number_generator.normal(
        0.0,
        np.sqrt(1.0 / input_dimension),
        size=(hidden_dimension, input_dimension),
    ).astype(np.float32)

    a = np.zeros((hidden_dimension,), dtype=np.float32)

    B = random_number_generator.normal(
        0.0,
        
        np.sqrt(1.0 / hidden_dimension),
        size=(output_dimension, hidden_dimension),
    ).astype(np.float32)

    b = np.zeros((output_dimension,), dtype=np.float32)

    return A, a, B, b

The gradient descent algorithm will update the parameters $A, \vec{a}, B$ and $\vec{b}$ using the rules:
\begin{align*}
A_{ij} & \leftarrow A_{ij} - \alpha \frac{\partial F}{\partial A_{ij}}, &\qquad
a_i & \leftarrow a_i - \alpha \frac{\partial F}{\partial a_i}, \\
B_{ij} & \leftarrow B_{ij} - \alpha \frac{\partial F}{\partial B_{ij}}, &\qquad
b_i & \leftarrow b_i - \alpha \frac{\partial F}{\partial b_i}.
\end{align*}
Note, here we are not tracking the iteration numbers in this notation.

The values of the parameters are well hidden in our definition of both the neural network operator and our objective function! But we can see that there are lots of compositions going on that need resolving!

In [ ]:
def sigma_prime_from_value(sigma_value):
    # derivative of sigmoid when the sigmoid value is already known
    return sigma_value * (1.0 - sigma_value)



In [ ]:
# ----------------------------
# Gradient descent
# ----------------------------
def train_network(x, targets, test_inputs, test_targets):
    random_number_generator = np.random.default_rng(seed)

    A, a, B, b = initialise_parameters(random_number_generator)

    L = indicator_matrix(targets, output_dimension)
    N = x.shape[1]

    accuracy_measures = {
        "objective": [],
        "test objective": [],
        "test accuracy": [],
    }

    for _ in range(1, iterations + 1):
        # Forward map
        (_, y, _, z) = forward_map(x, A, a, B, b)

        # --------------------------------------------------------
        # Gradient via chain rule (backpropagation)
        # --------------------------------------------------------
        sigma_p_z = sigma_prime_from_value(z)
        sigma_p_y = sigma_prime_from_value(y)

        dF_ds = ((z - L) * sigma_p_z) / N

        dF_dB = dF_ds
        @ y.T
        dF_db = np.sum(dF_ds, axis=1)

        dF_dy = B.T @ dF_ds
        dF_dr = dF_dy * sigma_p_y

        dF_dA = dF_dr @ x.T
        dF_da = np.sum(dF_dr, axis=1)
        

        # --------------------------------------------------------
        # Gradient descent step
        # --------------------------------------------------------
        A = A - alpha * dF_dA
        a = a - alpha * dF_da
        B = B - alpha * dF_dB
        b = b - alpha * dF_db

        # test on training data
        (_, _, _, z) = forward_map(x, A, a, B, b)
        current_objective = objective(z, targets)

        # test on test data
        test_objective, test_accuracy = evaluate(
            test_inputs, test_targets, A, a, B, b
        )

        accuracy_measures["objective"].append(current_objective)
        accuracy_measures["test objective"].append(test_objective)
        accuracy_measures["test accuracy"].append(100.0 * test_accuracy)

    return (A, a, B, b, accuracy_measures)

## Testing it out

We set some important parameters for the algorithm

In [ ]:
input_dimension = 28 * 28
hidden_dimension = 64
output_dimension = 10

iterations = 1000  # epochs
alpha = 8.0

# learning rate
seed = 42  # to help with initialisation


We also use an extra function to pick out the output $\vec{z}$ with the highest value to make our prediction of the final output value:

In [ ]:
# | echo: True
def predict(x, A, a, B, b):
    _, _, _, z = forward_map(x, A, a, B, b)
    return np.argmax(z, axis=0)


We can now do the training:

In [ ]:
# | echo: True
training_inputs, training_targets, test_inputs, test_targets = load_mnist()

# take transpose so that each column is a flattened image
training_inputs = training_inputs.T
test_inputs = test_inputs.T

print("training inputs shape:", training_inputs.shape)
print("training targets shape:", training_targets.shape)
print("test inputs shape:    ", test_inputs.shape)
print("test targets shape:   ", test_targets.shape)
print()



start = time.perf_counter()

A, a, B, b, accuracy_measures = train_network(
    training_inputs,
    training_targets,
    test_inputs,
    test_targets,
)

end = time.perf_counter()

print(f"completed {iterations} iterations in time {end - start:.4f}s.")

Let's see how well we've done:

In [ ]:
plt.semilogy(accuracy_measures["objective"], label="objective")
plt.semilogy(accuracy_measures["test objective"], label="test objective")
plt.grid(True)
plt.xlabel("Iterations")
plt.legend()
plt.show()

final_train_objective = accuracy_measures["objective"][-1]



final_test_objective = accuracy_measures["test objective"][-1]
final_test_accuracy = accuracy_measures["test accuracy"][-1]

print(f"Final train objective: {final_train_objective:.6f}")
print(f"Final test objective:  {final_test_objective:.6f}")
print(f"Final test accuracy:   {final_test_accuracy:.2f}%")

We see that we are achieving more than 90% accuracy within 1000 iterations and we have not converged to the best solution yet.

Let's see some examples of the classes and our predictions

In [ ]:
first_predictions = predict(test_inputs[:, :20], A, a, B, b)

# Show the first 10 test images
figure, axes = plt.subplots(2, 5)

for i, ax in enumerate(axes.flatten()):
    
    image = test_inputs[:, i].reshape(28, 28)
    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"Label: {test_targets[i]}\nPrediction: {first_predictions[i]}"
        
    )
    ax.axis("off")
    

plt.tight_layout()
plt.show()

I don't think this is too bad!

For one incorrect prediction (predicted 6 instead of 5), we can also check all $\vec{z}$ values that have:

In [ ]:
i = 7

x_single = test_inputs[:, [i]]  # shape (784, 1)

r, y, s, z = forward_map(x_single, A, a, B, b)



print("z values:")
for j in range(10):
    print(f"{j}: {z[j, 0]:.6f}")

We see that we were quite confident in our prediction and weren't close to predicting 5!

## Summary and outlook

We have demonstrated the simplest possible approach to training a neural network for the task for recognising hand-written numbers.

There are a few changes recommended to improve the performance of our neural network.

1. Replace the second sigmoid function by a *softmax*. This would mean $\vec{z} = \vec{z}(\vec{s})$ is given by:
   $$
   z_i(\vec{s}) = \frac{\exp(s_i)}{\sum_{j=0}^9 \exp(s_j)},
   $$
   and replace the L2 loss function by a *cross-entropy loss*. Then $J$ would be given by:
   $$
   J(\vec{x}) = - \sum_{i=0}^9 L_i(\vec{x}) \log z_i.
   $$

3. Replace the first sigmoid with a different activation function such as *ReLU*:
   $$
   y_k = \max(0, r_k).
   $$

4. Add one or more network layers...